# (g,s)-Dependent LOLR Test: $\sigma = 4\%$, $\Delta(g_L,s_B)=15\%$, $\Delta=0\%$ elsewhere

In [11]:
using Statistics
using Serialization
using Plots
include("src/simple_s_lolr_v2.jl")
Threads.nthreads()

8

## Scenario

In [12]:
# Scenario test notebook.
# LOLR spread is σ = 4% in all states.
# Δ is large only in (no default, g_L, s_B): 15%.
# Δ = 0% in all other (d, g, s) states.
Δ_dgs = fill(0.00, 2, 2, 2)
Δ_dgs[1, 1, 1] = 0.15
model = init_model(Model(
    Nb = 800,
    Nl = 20,
    Ne = 5,
    R_l = 1.075,
    Δ_dgs = Δ_dgs,
))
model.max_iter = 50
model.max_iter_vd = 100
model.max_iter_x = 100
model.pub = 0.65
model


Model(800, 20, 2, 2, 5, -0.05, 1.0, 0.0, 0.15138461538461534, [-0.05, -0.048685857321652065, -0.04737171464330413, -0.0460575719649562, -0.04474342928660826, -0.04342928660826033, -0.04211514392991239, -0.04080100125156445, -0.03948685857321652, -0.038172715894868585  …  0.9881727158948685, 0.9894868585732165, 0.9908010012515645, 0.9921151439299124, 0.9934292866082604, 0.9947434292866083, 0.9960575719649561, 0.9973717146433041, 0.998685857321652, 1.0], [0.0, 0.007967611336032387, 0.015935222672064774, 0.02390283400809716, 0.03187044534412955, 0.03983805668016193, 0.04780566801619432, 0.0557732793522267, 0.0637408906882591, 0.07170850202429148, 0.07967611336032386, 0.08764372469635624, 0.09561133603238864, 0.10357894736842102, 0.1115465587044534, 0.1195141700404858, 0.1274817813765182, 0.13544939271255058, 0.14341700404858296, 0.15138461538461534], [0.96, 1.04], [-2.5, -1.25, 0.0, 1.25, 2.5], [0.6 0.4; 0.25 0.75], [0.25 0.75; 0.25 0.75], [0.030396361765261396, 0.2355891672834392, 0.4680

## Solve Model

In [13]:
sol = solve_model(model; verbose = true)
println("Mean default probability in the state grid = ", mean(sol.d))
println("Final outer error = ", sol.outer_errs[end])

iter=1, vnd_err=2.9979732201660063, vd_err=6.586529810448383e-7, x_err=8.912555766737995e-7, damp=0.9
iter=2, vnd_err=5.048386118261279, vd_err=8.501175638997438e-7, x_err=0.00015225252810725787, damp=0.9
iter=3, vnd_err=4.942519785012012, vd_err=7.578296505883486e-7, x_err=1.3971139291621415e-5, damp=0.9
iter=4, vnd_err=0.5020746138788006, vd_err=9.235571827304057e-7, x_err=5.872815684593302e-6, damp=0.9
iter=5, vnd_err=0.7863354064645165, vd_err=7.090087184025151e-7, x_err=7.654389838768205e-6, damp=0.9
iter=6, vnd_err=0.3905661098991793, vd_err=8.921308278786455e-7, x_err=7.265106892528905e-6, damp=0.9
iter=7, vnd_err=0.4591870885865479, vd_err=7.590220718611818e-7, x_err=6.645082234746358e-6, damp=0.9
iter=8, vnd_err=0.22045481595253413, vd_err=8.894691099214924e-7, x_err=2.1042615682420607e-6, damp=0.9
iter=9, vnd_err=0.3432069305958141, vd_err=6.812370951791991e-7, x_err=4.62267371267866e-6, damp=0.9
iter=10, vnd_err=0.13680471461225707, vd_err=6.972600221644143e-7, x_err=1.57104

## Low-Growth Policy Plots

In [14]:

# Low-growth plots for the current scenario.
gi_low = argmin(model.g)
ei_mid = cld(model.Ne, 2)
s_bad = 1
s_good = model.Ns
li_zero = argmin(abs.(model.l))
lprime_zero = li_zero

mask_bad = sol.schedule_mask[:, lprime_zero, gi_low, s_bad]
mask_good = sol.schedule_mask[:, lprime_zero, gi_low, s_good]
schedule_bad = (n = sol.n[mask_bad, lprime_zero, gi_low, s_bad], R = sol.R[mask_bad, lprime_zero, gi_low, s_bad])
schedule_good = (n = sol.n[mask_good, lprime_zero, gi_low, s_good], R = sol.R[mask_good, lprime_zero, gi_low, s_good])

p_schedule = plot(
    schedule_bad.n,
    schedule_bad.R .- 1,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "private issuance n",
    ylabel = "private rate R - 1",
    title = "Private schedule at l' = 0 (σ = 4%, Δ(g_L,s_B)=15%, Δ=0 elsewhere) ",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (1100, 650),
)
plot!(p_schedule, schedule_good.n, schedule_good.R .- 1, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

bad_bp = sol.b_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
bad_lp = sol.l_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
good_bp = sol.b_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
good_lp = sol.l_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
bad_n = [sol.n[bad_bp[bi], bad_lp[bi], gi_low, s_bad] for bi in 1:model.Nb]
good_n = [sol.n[good_bp[bi], good_lp[bi], gi_low, s_good] for bi in 1:model.Nb]
bad_repay = .!sol.d[:, li_zero, gi_low, s_bad, ei_mid]
good_repay = .!sol.d[:, li_zero, gi_low, s_good, ei_mid]

p_nb = plot(
    model.b[bad_repay],
    bad_n[bad_repay],
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "private issuance n",
    title = "Private issuance policy n(b) (σ = 4%, Δ(g_L,s_B)=15%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (1100, 650),
)
plot!(p_nb, model.b[good_repay], good_n[good_repay], color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

bad_lp_nd = sol.l_policy_idx[:, li_zero, gi_low, s_bad, ei_mid]
good_lp_nd = sol.l_policy_idx[:, li_zero, gi_low, s_good, ei_mid]
bad_lp_d = sol.l_policy_idx_d[:, li_zero, gi_low, s_bad, ei_mid]
good_lp_d = sol.l_policy_idx_d[:, li_zero, gi_low, s_good, ei_mid]
bad_nl_nd = (model.g[gi_low] .* model.l[bad_lp_nd]) ./ model.R_l
good_nl_nd = (model.g[gi_low] .* model.l[good_lp_nd]) ./ model.R_l
bad_nl_d = (model.g[gi_low] .* model.l[bad_lp_d]) ./ model.R_l
good_nl_d = (model.g[gi_low] .* model.l[good_lp_d]) ./ model.R_l

p_nl_nd = plot(
    model.b,
    bad_nl_nd,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "LOLR issuance n_l",
    title = "ND policy n_l(b) (σ = 4%, Δ(g_L,s_B)=15%, Δ=0 elsewhere, l = 0)",
    legend = :topright,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (700, 600),
)
plot!(p_nl_nd, model.b, good_nl_nd, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

p_nl_d = plot(
    model.b,
    bad_nl_d,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "LOLR issuance n_l",
    title = "D policy n_l(b) (σ = 4%, Δ(g_L,s_B)=15%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    size = (700, 600),
)
plot!(p_nl_d, model.b, good_nl_d, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")
p_nl = plot(p_nl_nd, p_nl_d, layout = (1, 2), size = (1350, 550))

d_bad = Float64.(sol.d[:, li_zero, gi_low, s_bad, ei_mid])
d_good = Float64.(sol.d[:, li_zero, gi_low, s_good, ei_mid])
p_default = plot(
    model.b,
    d_bad,
    color = :firebrick,
    linestyle = :solid,
    linewidth = 2.5,
    label = "bad s",
    xlabel = "current private debt b",
    ylabel = "default decision d",
    title = "Default policy d(b) (σ = 4%, Δ(g_L,s_B)=15%, Δ=0 elsewhere, l = 0)",
    legend = :topleft,
    legendfontsize = 10,
    left_margin = 12Plots.mm,
    bottom_margin = 10Plots.mm,
    ylims = (-0.05, 1.05),
    size = (1100, 650),
)
plot!(p_default, model.b, d_good, color = :steelblue, linestyle = :dash, linewidth = 2.5, label = "good s")

schedule_path = joinpath("result", "s_lolr_v2_sigma4_delta15_else0_" * "low_growth_selected_schedule.png")
nb_path = joinpath("result", "s_lolr_v2_sigma4_delta15_else0_" * "low_growth_private_policy.png")
nl_path = joinpath("result", "s_lolr_v2_sigma4_delta15_else0_" * "low_growth_lolr_policy.png")
default_path = joinpath("result", "s_lolr_v2_sigma4_delta15_else0_" * "low_growth_default_policy.png")

savefig(p_schedule, schedule_path)
savefig(p_nb, nb_path)
savefig(p_nl, nl_path)
savefig(p_default, default_path)

(
    schedule_path = schedule_path,
    private_policy_path = nb_path,
    lolr_policy_path = nl_path,
    default_path = default_path,
)


(schedule_path = "result/s_lolr_v2_sigma4_delta15_else0_low_growth_selected_schedule.png", private_policy_path = "result/s_lolr_v2_sigma4_delta15_else0_low_growth_private_policy.png", lolr_policy_path = "result/s_lolr_v2_sigma4_delta15_else0_low_growth_lolr_policy.png", default_path = "result/s_lolr_v2_sigma4_delta15_else0_low_growth_default_policy.png")

## Save Solved Scenario Bundle

In [15]:

# Save the solved scenario bundle for later comparison tables.
scenario_meta = (
    name = "σ=4, ΔLB=15, Δelse=0",
    header_top = raw"S4-15/0",
    header_bottom = raw"$\sigma=4\%,\ \Delta_{ND,LB}=15\%,\ \Delta_{else}=0\%$",
    bundle_name = "s_lolr_v2_case_sigma4_delta15_else0.jls",
)

bundle_path = joinpath("result", scenario_meta.bundle_name)
mkpath(dirname(bundle_path))
open(bundle_path, "w") do io
    serialize(io, (scenario = scenario_meta, model = model, sol = sol))
end
println(bundle_path)


result/s_lolr_v2_case_sigma4_delta15_else0.jls
